# Pictionary Night: A CNN Settles It
### CSCI 4220 · in-class coding · *UDL ch. 10 — Convolutional networks*

At department game night, Robby drew a cat. Dr. Dan guessed **pig**. Dr. Sanders guessed
**smiley face**. Robby's team lost, and Robby has appealed.

The judge: a convolutional network we build from scratch in numpy and train on **25,000 real
doodles** from Google's *Quick, Draw!*, a game in which people around the world drew things in under 20 seconds.

| min | | you write |
|---|---|---|
| 0–10 | **1** · real data from the internet, and what's wrong with it | scaling, checks |
| 10–18 | **2** · convolution by hand: a whisker detector (slides 56–58, 11–12) | `conv2d` |
| 18–25 | **3** · the CNN (slides 26–31, 36–40, 63–65) | `maxpool2`, `cnn_predict` |
| 25–37 | **4** · train both networks, read the results | accuracy, a confusion matrix |
| 37–44 | **5** · look inside, and the verdict (slides 42–45) | the heat map |
| 44–60 | **6** · the appeal: three rounds | no new code: predict, then run |

**Blanks:** type the commented line directly above each `______`.

▶ **Run these two cells once**: imports, then the helpers (download, drawing, plotting, the fast convolution, training).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ── helpers: run once, never edit ───────────────────────────────────────────────
import os, time, urllib.request, urllib.parse
from PIL import Image, ImageDraw
np.seterr(all="ignore")
plt.rcParams.update({"axes.grid": False, "font.size": 11, "axes.titleweight": "bold", "axes.titlelocation": "left"})
swv = np.lib.stride_tricks.sliding_window_view

# ── the data: Google's Quick, Draw! dataset (quickdraw.withgoogle.com/data, CC BY 4.0) ──
CLASSES = ["cat", "dog", "pig", "owl", "lion", "sun", "The Mona Lisa", "smiley face", "hot dog", "banana"]
QD_URL = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/{}.npy"

def quickdraw(name, n=2500):
    """The first n drawings of one category, (n, 28, 28) uint8: 0 = paper, 255 = ink.
    Downloads only those bytes from Google (about 2 MB), once; after that it reads ./quickdraw/."""
    path = os.path.join("quickdraw", f"{name}_{n}.npy")
    if not os.path.exists(path):
        url, hdr = QD_URL.format(urllib.parse.quote(name)), {"Range": f"bytes=0-{255 + 784 * n}"}
        try:
            import requests
            raw = requests.get(url, headers=hdr, timeout=60).content
        except ImportError:
            raw = urllib.request.urlopen(urllib.request.Request(url, headers=hdr), timeout=60).read()
        assert raw[:6] == b"\x93NUMPY", f"download of {name!r} failed: {raw[:80]!r}"
        start = 10 + int.from_bytes(raw[8:10], "little")              # skip the .npy header
        os.makedirs("quickdraw", exist_ok=True)
        np.save(path, np.frombuffer(raw[start:start + 784 * n], np.uint8).reshape(n, 28, 28))
    return np.load(path)

def split(X, y, test=0.2, seed=0):
    """Shuffle, then hold out a fraction of the drawings for testing."""
    order = np.random.default_rng(seed).permutation(len(X)); k = int(len(X) * test)
    return X[order[k:]], y[order[k:]], X[order[:k]], y[order[:k]]

def nudge(X, px):
    """Every drawing moved px pixels right and px pixels down (blank paper fills the gap)."""
    out = np.zeros_like(X); out[:, px:, px:] = X[:, :X.shape[1] - px, :X.shape[2] - px]
    return out

# ── the pencil: strokes are lists of (x, y) points, x to the right, y DOWN ──
def circle(cx, cy, r, ry=None, start=0, end=360):
    """A circle, an ellipse (give ry), or an arc from start to end degrees (0 = right, 90 = down)."""
    t = np.radians(np.linspace(start, end, 24)); ry = r if ry is None else ry
    return list(zip(cx + r * np.cos(t), cy + ry * np.sin(t)))

def render(strokes, width=13):
    """Strokes -> a 28x28x1 drawing, scaled and centred the way Quick, Draw! prepares its own."""
    pts = np.concatenate([np.array(s, float) for s in strokes]); lo = pts.min(0); size = pts.max(0) - lo
    span = max(size.max(), 1e-9); off = (span - size) / 2
    im = Image.new("L", (256, 256), 0); d = ImageDraw.Draw(im)
    for s in strokes:
        q = [tuple(16 + (np.array(p, float) - lo + off) * 224 / span) for p in s]
        d.line(q * (2 if len(q) == 1 else 1), fill=255, width=width, joint="curve")
        for x, y in (q[0], q[-1]): d.ellipse([x - width / 2, y - width / 2, x + width / 2, y + width / 2], fill=255)
    return np.asarray(im.resize((28, 28), Image.BOX), np.float32)[..., None] / 255

# ── ready-made parts (all drawn on the same 100 x 100 sheet) ──
face        = [circle(50, 55, 30)]
round_ears  = [circle(36, 30, 8, start=160, end=380), circle(64, 30, 8, start=160, end=380)]
pointy_ears = [[(28, 40), (30, 15), (44, 28)], [(56, 28), (70, 15), (72, 40)]]
eyes        = [[(40, 47)], [(60, 47)]]
snout       = [circle(50, 64, 11, 8), [(46, 64)], [(54, 64)]]
nose        = [[(47, 60), (53, 60), (50, 64), (47, 60)]]
smile       = [[(42, 70), (46, 74), (50, 70), (54, 74), (58, 70)]]
whiskers    = [[(15, 60), (38, 62)], [(15, 70), (38, 67)], [(62, 62), (85, 60)], [(62, 67), (85, 70)]]
PARTS = dict(face=face, round_ears=round_ears, pointy_ears=pointy_ears, eyes=eyes, snout=snout, nose=nose, smile=smile, whiskers=whiskers)
def sketch(*parts):
    """Put parts together into one 28x28x1 drawing."""
    return render([stroke for part in parts for stroke in part])
def show_parts():
    fig, ax = plt.subplots(1, len(PARTS), figsize=(1.6 * len(PARTS), 1.9))
    for a, (name, part) in zip(ax, PARTS.items()):
        for s in face: a.plot(*np.array(s).T, color="#ddd", lw=2)
        for s in part:
            q = np.array(s); a.plot(*q.T, color="k", lw=2.5, marker="o" if len(q) == 1 else None, ms=4)
        a.set_xlim(10, 90); a.set_ylim(90, 10); a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([]); a.set_title(name, fontsize=9)
    fig.tight_layout(); plt.show()

# ── plots ──
def _nm(k): return CLASSES[k].replace("The ", "")
def _ink(ax, img, title=""):
    ax.imshow(1 - np.squeeze(img).astype(np.float32) / (255 if np.max(img) > 1 else 1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
def show_drawings(X, titles=None, cols=8):
    n = len(X); rows = -(-n // cols); cols = min(cols, n); s = 2.8 if n == 1 else 1.9
    fig, ax = plt.subplots(rows, cols, figsize=(s * cols, (s + .3) * rows), squeeze=False)
    for k, a in enumerate(ax.ravel()):
        if k < n: _ink(a, X[k], titles[k] if titles is not None else "")
        else: a.axis("off")
    fig.tight_layout(); plt.show()
def show_classes(raw, n=10):
    fig, ax = plt.subplots(len(raw), n, figsize=(1.05 * n + 1.2, 1.05 * len(raw)))
    for r, (name, X) in enumerate(raw.items()):
        for c in range(n): _ink(ax[r, c], X[c * 7])
        ax[r, 0].set_ylabel(name, rotation=0, ha="right", va="center", fontsize=10)
    fig.tight_layout(); plt.show()
def show_maps(maps, titles, img=None, cols=8):
    items = ([("drawing", None)] if img is not None else []) + list(zip(titles, maps))
    rows = -(-len(items) // cols); cols = min(cols, len(items))
    fig, ax = plt.subplots(rows, cols, figsize=(2.1 * cols, 2.3 * rows), squeeze=False)
    for a, (t, m) in zip(ax.ravel(), items):
        if m is None: _ink(a, img, t)
        else: a.imshow(np.squeeze(m), cmap="magma"); a.set_title(t, fontsize=9); a.set_xticks([]); a.set_yticks([])
    for a in ax.ravel()[len(items):]: a.axis("off")
    fig.tight_layout(); plt.show()
def show_filters(K0):
    F = K0.shape[3]; fig, ax = plt.subplots(2, F // 2, figsize=(1.2 * F // 2, 2.8))
    for f, a in enumerate(ax.ravel()):
        a.imshow(K0[:, :, 0, f], cmap="RdBu_r", vmin=-np.abs(K0).max(), vmax=np.abs(K0).max())
        a.set_title(f"{f}", fontsize=8); a.set_xticks([]); a.set_yticks([])
    fig.suptitle("first-layer filters (red = wants ink, blue = wants paper)", x=.02, ha="left", fontsize=10, fontweight="bold"); plt.show()
def show_confusion(y_true, y_pred):
    M = np.zeros((len(CLASSES), len(CLASSES)), int); np.add.at(M, (y_true, y_pred), 1)
    fig, ax = plt.subplots(figsize=(7.4, 6.2)); ax.imshow(M / M.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            if M[i, j]: ax.text(j, i, M[i, j], ha="center", va="center", fontsize=8, color="white" if M[i, j] > .5 * M[i].sum() else "#333")
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=60, ha="right"); ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel("the network said"); ax.set_ylabel("the player was asked to draw"); plt.show()
def show_wrong(P, X, y, n=8):
    p = predict(P, X); guess = p.argmax(1); wrong = np.flatnonzero(guess != y)
    worst = wrong[np.argsort(-p[wrong, guess[wrong]])][:n]
    show_drawings(X[worst], [f"asked: {_nm(y[i])}\nsaid: {_nm(guess[i])} {p[i, guess[i]]:.2f}" for i in worst])
def reveal(P, X, y, ix):
    p = predict(P, X[ix])
    show_drawings(X[ix], [f"#{k + 1} {_nm(y[i])}\nnet: {_nm(p[k].argmax())} {p[k].max():.2f}" for k, i in enumerate(ix)])
def show_verdict(img, p, title="the verdict"):
    o = np.argsort(p); fig, ax = plt.subplots(1, 2, figsize=(8, 3.2), gridspec_kw={"width_ratios": [1, 1.6]})
    _ink(ax[0], img, title)
    ax[1].barh(range(len(p)), p[o], color=["#c0392b" if k == o[-1] else "#9aa5b1" for k in o])
    ax[1].set_yticks(range(len(p)), [_nm(k) for k in o]); ax[1].set_xlim(0, 1)
    for s in ("top", "right"): ax[1].spines[s].set_visible(False)
    ax[1].set_title(f"{_nm(o[-1])}: {p[o[-1]]:.2f}"); fig.tight_layout(); plt.show()
def _overlay(ax, img, h, top, title):
    big = np.asarray(Image.fromarray(np.uint8(255 * np.clip(h / top, 0, 1))).resize((28, 28), Image.BILINEAR)) / 255
    rgba = plt.cm.YlOrRd(.25 + .75 * big); rgba[..., 3] = np.clip(1.3 * big, 0, .85)   # no evidence = clear
    ax.imshow(1 - .5 * np.squeeze(img), cmap="gray", vmin=0, vmax=1); ax.imshow(rgba)
    ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
def show_heat(img, heats, titles):
    top = max(1.0, max(float(h.max()) for h in heats))           # one shared scale, so a weak response looks weak
    fig, ax = plt.subplots(1, len(heats) + 1, figsize=(2.9 * (len(heats) + 1), 3.2))
    _ink(ax[0], img, "drawing")
    for a, h, t in zip(ax[1:], heats, titles): _overlay(a, img, h, top, t)
    fig.tight_layout(); plt.show()

# ── fast batched convolution: the same sum-of-products as your loop, for many images and channels at once ──
def _conv_cols(x, k):
    p = k // 2; N, H, W, Cin = x.shape
    return np.ascontiguousarray(swv(np.pad(x, ((0, 0), (p, p), (p, p), (0, 0))), (k, k), axis=(1, 2)).transpose(0, 1, 2, 4, 5, 3)).reshape(N * H * W, k * k * Cin)
def conv(x, K, b):
    """x: (N, H, W, C_in)   K: (k, k, C_in, C_out)   b: (C_out,)   ->  (N, H, W, C_out), zero padding keeps H x W."""
    N, H, W, _ = x.shape
    return (_conv_cols(x, K.shape[0]) @ K.reshape(-1, K.shape[3]) + b).reshape(N, H, W, K.shape[3])

# ── training: Adam on cross-entropy (PS2, now with 10 classes); backprop through each layer as in IC 4 ──
def _conv_back(g, x, K, cols):
    # weight gradient = input patches x upstream gradient; input gradient = each output's blame sent back through K
    N, H, W, Cin = x.shape; k, F = K.shape[0], K.shape[3]; p = k // 2; g2 = g.reshape(-1, F)
    dK, db = (cols.T @ g2).reshape(K.shape), g2.sum(0)
    dcols = (g2 @ K.reshape(-1, F).T).reshape(N, H, W, k, k, Cin); dxp = np.zeros((N, H + 2 * p, W + 2 * p, Cin), np.float32)
    for i in range(k):
        for j in range(k): dxp[:, i:i + H, j:j + W, :] += dcols[:, :, :, i, j, :]
    return dxp[:, p:p + H, p:p + W, :], dK, db

def init_cnn(seed=0, channels=(1, 16, 32, 64), sizes=(5, 3, 3)):
    rng = np.random.default_rng(seed); P = {}
    for i, (a, b) in enumerate(zip(channels[:-1], channels[1:])):
        k = sizes[i]; P[f"K{i}"] = (rng.standard_normal((k, k, a, b)) * np.sqrt(2 / (k * k * a))).astype(np.float32); P[f"b{i}"] = np.zeros(b, np.float32)
    P["w"] = (rng.standard_normal((channels[-1], len(CLASSES))) * np.sqrt(1 / channels[-1])).astype(np.float32); P["c"] = np.zeros(len(CLASSES), np.float32)
    return P
def init_dense(seed=0, d_in=28 * 28, hidden=64):
    rng = np.random.default_rng(seed)
    return {"W1": (rng.standard_normal((d_in, hidden)) * np.sqrt(2 / d_in)).astype(np.float32), "b1": np.zeros(hidden, np.float32),
            "W2": (rng.standard_normal((hidden, len(CLASSES))) * np.sqrt(1 / hidden)).astype(np.float32), "b2": np.zeros(len(CLASSES), np.float32)}

def _softmax(z):
    e = np.exp(z - z.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)
def _cnn_fb(P, x, y=None):
    cache, h = [], x
    for i in range(3):
        K = P[f"K{i}"]; cols = _conv_cols(h, K.shape[0]); z = (cols @ K.reshape(-1, K.shape[3]) + P[f"b{i}"]).reshape(h.shape[:3] + (-1,))
        a = np.maximum(z, 0); e = dict(x=h, z=z, cols=cols)
        if i < 2:
            N, H, W, F = a.shape; r = a.reshape(N, H // 2, 2, W // 2, 2, F); a2 = r.max((2, 4)); e["mask"] = r == a2[:, :, None, :, None, :]; a = a2
        cache.append(e); h = a
    m = h.max((1, 2)); logit = m @ P["w"] + P["c"]
    if y is None: return logit
    dl = _softmax(logit); dl[np.arange(len(y)), y] -= 1; dl /= len(y); G = {"w": m.T @ dl, "c": dl.sum(0)}
    g = (h == m[:, None, None, :]) * (dl @ P["w"].T)[:, None, None, :]
    for i in (2, 1, 0):
        e = cache[i]
        if "mask" in e:
            N, hh, _, ww, _, F = e["mask"].shape; g = (e["mask"] * g[:, :, None, :, None, :]).reshape(N, hh * 2, ww * 2, F)
        g = g * (e["z"] > 0); g, G[f"K{i}"], G[f"b{i}"] = _conv_back(g, e["x"], P[f"K{i}"], e["cols"])
    return logit, G
def _dense_fb(P, x, y=None):
    f = x.reshape(len(x), -1); z = f @ P["W1"] + P["b1"]; h = np.maximum(z, 0); logit = h @ P["W2"] + P["b2"]
    if y is None: return logit
    dl = _softmax(logit); dl[np.arange(len(y)), y] -= 1; dl /= len(y); G = {"W2": h.T @ dl, "b2": dl.sum(0)}
    g = (dl @ P["W2"].T) * (z > 0); G["W1"], G["b1"] = f.T @ g, g.sum(0)
    return logit, G

def train(kind, X, y, epochs=None, lr=None, batch=64, seed=0):
    """kind: "cnn" or "dense". Mini-batch Adam on cross-entropy. Prints the loss after each epoch."""
    cnn = kind == "cnn"; epochs = epochs or (6 if cnn else 8); lr = lr or (3e-3 if cnn else 1e-3)
    P = init_cnn(seed) if cnn else init_dense(seed); fb = _cnn_fb if cnn else _dense_fb
    m = {k: np.zeros_like(v) for k, v in P.items()}; v = {k: np.zeros_like(v) for k, v in P.items()}; t = 0; rng = np.random.default_rng(seed); t0 = time.time()
    for ep in range(epochs):
        losses = []
        for bb in np.array_split(rng.permutation(len(X)), len(X) // batch):
            logit, G = fb(P, X[bb], y[bb]); t += 1
            losses.append(-np.mean(np.log(_softmax(logit)[np.arange(len(bb)), y[bb]] + 1e-9)))
            for k in P:
                m[k] = .9 * m[k] + .1 * G[k]; v[k] = .999 * v[k] + .001 * G[k] ** 2
                P[k] -= lr * (m[k] / (1 - .9 ** t)) / (np.sqrt(v[k] / (1 - .999 ** t)) + 1e-8)
        print(f"  {kind:5s} epoch {ep + 1}   loss {np.mean(losses):.3f}   ({time.time() - t0:.0f} s)")
    P["kind"] = kind
    return P

def predict(P, X):
    """Class probabilities for every drawing, (N, 10) (uses the trainer's own forward pass)."""
    fb = _cnn_fb if P["kind"] == "cnn" else _dense_fb
    return np.concatenate([_softmax(fb(P, X[s:s + 500])) for s in range(0, len(X), 500)])

def _heat(P, img, c):
    h = img[None]
    for i in range(3):
        h = np.maximum(conv(h, P[f"K{i}"], P[f"b{i}"]), 0)
        if i < 2: N, H, W, F = h.shape; h = h.reshape(N, H // 2, 2, W // 2, 2, F).max((2, 4))
    return np.maximum(h[0] @ P["w"][:, c], 0)
def judge(P, *parts):
    """The judge's ruling on some parts (or on one 28x28x1 drawing): its top 5 guesses, and where it sees a cat."""
    img = parts[0] if isinstance(parts[0], np.ndarray) else sketch(*parts)
    p = predict(P, img[None])[0]; o = list(np.argsort(p)[::-1][:5]); rival = o[0] if o[0] != 0 else o[1]
    if 0 not in o: o[-1] = 0                                      # cat always gets a bar
    hc, hr = _heat(P, img, 0), _heat(P, img, rival); top = max(1.0, hc.max(), hr.max())
    fig, ax = plt.subplots(1, 4, figsize=(11.5, 2.9), gridspec_kw={"width_ratios": [1, 1.25, 1, 1]})
    _ink(ax[0], img, f"ruling: {_nm(int(np.argmax(p)))} ({p.max():.2f})")
    ax[1].barh(range(5), p[o][::-1], color=["#c0392b" if k == 0 else "#9aa5b1" for k in o[::-1]])
    ax[1].set_yticks(range(5), [_nm(k) for k in o[::-1]]); ax[1].set_xlim(0, 1); ax[1].set_title(f"cat: {p[0]:.2f}")
    for s in ("top", "right"): ax[1].spines[s].set_visible(False)
    _overlay(ax[2], img, hc, top, "where it sees a cat"); _overlay(ax[3], img, hr, top, f"where it sees a {_nm(rival)}")
    fig.tight_layout(); plt.show()

def try_it(fn):
    try:
        fn(); print("no error")
    except Exception as e:
        print(f"{type(e).__name__}: {e}")
print("helpers ready")

▶ Exhibit A: Robby's cat, rebuilt from the helpers' ready-made parts.

In [ ]:
robbys_cat = sketch(face, round_ears, snout, eyes)
show_drawings([robbys_cat], ["Robby's cat"])

**Question for class:** Pig, smiley face, or cat? Hands up.

---
## 1 · Real data from the internet, and what's wrong with it  <small>0–10 min</small>

▶ Download 2,500 drawings of each of 10 categories straight from Google *(about 20 MB, once)*.

In [ ]:
raw = {name: quickdraw(name) for name in CLASSES}

▶ Check what arrived: shape, type, range.

In [ ]:
# print(raw["cat"].shape, raw["cat"].dtype, raw["cat"].min(), raw["cat"].max())
______

**Question for class:** What does one row of `raw["cat"]` hold? What do 0 and 255 mean?

▶ Here is what the internet draws when it is asked for each category.

In [ ]:
show_classes(raw)

**Question for class:** Which category has the worst artists? Which two will the network mix up?

▶ Real data has junk in it. Sort every drawing by how much ink it uses and look at both ends.

In [ ]:
everything = np.concatenate(list(raw.values()))
asked = np.repeat(CLASSES, 2500)
ink = everything.mean(axis=(1, 2))
order = np.argsort(ink)
show_drawings(everything[np.r_[order[:8], order[-8:]]], list(asked[np.r_[order[:8], order[-8:]]]))

**Question for class:** Keep these in the training set, or throw them out? What would each choice cost?

**Takeaway:** the label is what the player was *asked* to draw, not what they drew. Some labels are wrong
and we can't know which ones, so even a perfect network can't score 100%.

▶ Build the training data: stack the classes, scale the ink to 0–1, add a channel axis, split off a test set.

In [ ]:
X = np.concatenate([raw[name] for name in CLASSES])          # (25000, 28, 28) uint8
y = np.repeat(np.arange(len(CLASSES)), 2500)                 # 0 = cat, 1 = dog, ...
# X = X.astype(np.float32)[..., None] / 255
X = ______
X_train, y_train, X_test, y_test = split(X, y, test=0.2)
print(X_train.shape, X_test.shape, X_train.dtype, X_train.max())

**Question for class:** Why `[..., None]`? *(How many colour channels does a doodle have?)*

---
## 2 · Convolution by hand: a whisker detector  <small>10–18 min · slides 56–58, 11–12</small>

A 2D convolution slides a small **kernel** $K$ over the image; each output is a weighted sum of the
patch underneath it (zero padding keeps the size the same):

$$h_{ij} = \sum_{m}\sum_{n} K_{mn}\; x_{i+m,\, j+n}$$

▶ Write it: two loops and one line of arithmetic.

In [ ]:
def conv2d(img, K):
    k = K.shape[0]
    padded = np.pad(img, k // 2)
    out = np.zeros_like(img)
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            # out[i, j] = np.sum(padded[i:i + k, j:j + k] * K)
            out[i, j] = ______
    return out

▶ Hand-design a **whisker detector** (ink in the middle row, paper above and below), and its
sideways twin, and run both on a real cat.

In [ ]:
doodle = X[296, :, :, 0]                                     # a cat from the dataset
whisker_filter = np.array([[-1, -1, -1],
                           [ 2,  2,  2],
                           [-1, -1, -1]], dtype=np.float32)
# whisker_map = conv2d(doodle, whisker_filter)
whisker_map = ______
# upright_map = conv2d(doodle, whisker_filter.T)
upright_map = ______
show_maps([np.maximum(whisker_map, 0), np.maximum(upright_map, 0)], ["whisker detector", "sideways twin"], img=doodle)

**Question for class:** What else lights up the whisker detector? Would it help tell a cat from a lion?

▶ The helpers' `conv` does the same sum for whole batches and many channels at once. Check that it matches yours.

In [ ]:
fast = conv(doodle[None, :, :, None], whisker_filter[:, :, None, None], np.zeros(1, np.float32))[0, :, :, 0]
print("same as your loop?", np.allclose(fast, whisker_map, atol=1e-5))

▶ **Equivariance** (slides 11–12): move the cat 4 pixels right, then filter; compare with filtering first, then moving.

In [ ]:
# shifted = np.roll(doodle, 4, axis=1)
shifted = ______
a = conv2d(shifted, whisker_filter)
b = np.roll(conv2d(doodle, whisker_filter), 4, axis=1)
print("move then filter == filter then move?", np.allclose(a[:, 6:-6], b[:, 6:-6], atol=1e-5))

**Takeaway:** the same kernel runs at every position, so whiskers that move in the drawing move the
same way in the map. A filter that finds whiskers finds them **wherever** they are.

---
## 3 · The CNN  <small>18–25 min · slides 26–31, 36–40, 63–65</small>

Three convolution layers with ReLU; **max pooling** halves the size after the first two; a **global max**
asks each of the 64 final channels *"did you see your feature anywhere?"*; then one score per class.

$$28{\times}28{\times}1 \xrightarrow{5\times5\ \text{conv, ReLU, pool}} 14{\times}14{\times}16 \xrightarrow{\text{conv, ReLU, pool}} 7{\times}7{\times}32 \xrightarrow{\text{conv, ReLU}} 7{\times}7{\times}64 \xrightarrow{\text{max over positions}} 64 \xrightarrow{\text{weights, softmax}} 10\ \text{probabilities}$$

▶ Write ReLU and 2×2 max pooling (slide 64): split every 2×2 block out, keep the largest.

In [ ]:
def relu(z):
    # return np.maximum(0, z)
    return ______

def maxpool2(x):
    N, H, W, F = x.shape
    # return x.reshape(N, H // 2, 2, W // 2, 2, F).max(axis=(2, 4))
    return ______

▶ Write the forward pass, one line per layer. Softmax turns the 10 scores into probabilities.

In [ ]:
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def cnn_predict(P, X):
    h = maxpool2(relu(conv(X, P["K0"], P["b0"])))      # 28x28x1  -> 14x14x16
    h = maxpool2(relu(conv(h, P["K1"], P["b1"])))      # 14x14x16 -> 7x7x32
    # h = relu(conv(h, P["K2"], P["b2"]))
    h = ______                                         # 7x7x32   -> 7x7x64
    # evidence = h.max(axis=(1, 2))
    evidence = ______                                  # each channel's strongest response anywhere
    return softmax(evidence @ P["w"] + P["c"])

▶ Count parameters: this CNN vs. a dense network with one hidden layer of 64 units.

In [ ]:
cnn_params = init_cnn()
dense_params = init_dense()
# n_cnn = sum(v.size for v in cnn_params.values())
n_cnn = ______
# n_dense = sum(v.size for v in dense_params.values())
n_dense = ______
print(f"CNN: {n_cnn:,} parameters    dense: {n_dense:,} parameters")

**Question for class:** The CNN has 64 channels at the end and the dense network only 64 hidden units. Why is the CNN still smaller?

**Takeaway:** **weight sharing** (slides 26–31). Every position reuses the same small kernels, so
the parameter count depends on kernel size and channels, not on the size of the image.

▶ Normal error: hand the network one drawing instead of a batch.

In [ ]:
try_it(lambda: cnn_predict(cnn_params, robbys_cat))

**Takeaway:** the network expects `(N, H, W, C)`. For one drawing, add the batch axis: `robbys_cat[None]`.

---
## 4 · Train both networks, read the results  <small>25–37 min</small>

`train` runs mini-batch Adam on cross-entropy (PS2's loss, now over 10 classes). Its backward pass is
IC 4's, plus one new rule for convolution: *weight gradient = input patches × upstream gradient*.

▶ Train the dense network, then the CNN *(about 30 s; read the confusion matrix question below while it runs)*.

In [ ]:
# dense = train("dense", X_train, y_train)
dense = ______
# cnn = train("cnn", X_train, y_train)
cnn = ______

▶ Check that your `cnn_predict` gives exactly what the trainer's network computes.

In [ ]:
print("your forward pass matches:", np.allclose(cnn_predict(cnn, X_test[:500]), predict(cnn, X_test[:500]), atol=1e-5))

▶ Score both on the test drawings, and on the same drawings **nudged 3 pixels** right and down
(Quick, Draw! centres every drawing; a photo of a napkin sketch won't be).

In [ ]:
def accuracy(P, X, y):
    # return np.mean(predict(P, X).argmax(axis=1) == y)
    return ______

for name, P in (("dense", dense), ("CNN", cnn)):
    print(f"{name:6s} test: {accuracy(P, X_test, y_test):.2f}    nudged 3 px: {accuracy(P, nudge(X_test, 3), y_test):.2f}")

**Question for class:** Three pixels. What happened to the dense network, and why not to the CNN?

**Takeaway:** the dense network has a separate weight for every pixel, so it learned **where** ink
usually is. The CNN learned **what** strokes look like, and its filters run everywhere (slides 9–12).

▶ Where does the CNN go wrong? Rows: what the player was asked to draw. Columns: what the network said.

In [ ]:
# y_pred = predict(cnn, X_test).argmax(axis=1)
y_pred = ______
show_confusion(y_test, y_pred)

**Question for class:** Find the biggest off-diagonal number. Would *you* have made the same mistake?

▶ The CNN's most confident mistakes.

In [ ]:
show_wrong(cnn, X_test, y_test)

**Question for class:** For each one: is the network wrong, or is the label?

▶ **Humans vs. the network.** Write down your guess for each drawing.

In [ ]:
quiz = np.random.default_rng(15).choice(len(X_test), 8, replace=False)
show_drawings(X_test[quiz], [f"#{k + 1}" for k in range(8)])

▶ Reveal: the answers, and the network's guesses. Who scored more?

In [ ]:
reveal(cnn, X_test, y_test, quiz)

---
## 5 · Look inside, and the verdict  <small>37–44 min · slides 42–45</small>

▶ The 16 first-layer filters (5×5 each), and what each one sees in the cat from section 2.

In [ ]:
show_filters(cnn["K0"])
layer1 = relu(conv(doodle[None, :, :, None], cnn["K0"], cnn["b0"]))[0]
show_maps([layer1[:, :, f] for f in range(16)], [f"filter {f}" for f in range(16)], img=doodle, cols=9)

**Question for class:** Which filter did the network learn that you hand-designed in section 2? Which filters respond to the *paper* instead of the ink?

▶ **The verdict** on Robby's cat.

In [ ]:
show_verdict(robbys_cat, cnn_predict(cnn, robbys_cat[None])[0])

Before the global max, the last layer is a 7×7 grid of 64-channel responses. Weighting the channels
by one class's weights gives the **evidence for that class at every position**: a heat map nobody trained.

▶ Write the heat map, then ask the network *where* it sees each guest's answer.

In [ ]:
def heat_map(P, img, c):
    h = maxpool2(relu(conv(img[None], P["K0"], P["b0"])))
    h = maxpool2(relu(conv(h, P["K1"], P["b1"])))
    h = relu(conv(h, P["K2"], P["b2"]))[0]                   # 7x7x64
    # return np.maximum(h @ P["w"][:, c], 0)
    return ______

show_heat(robbys_cat, [heat_map(cnn, robbys_cat, CLASSES.index(c)) for c in ("pig", "smiley face", "cat")],
          ["where it sees a pig", "... a smiley face", "... a cat"])

**Question for class:** What part of the drawing convinced it? Who won game night, Dr. Dan or Dr. Sanders?

**Takeaway:** each layer-3 unit sees about a 20×20 patch of the drawing (its **receptive field**,
slides 42–45). The heat map shows which patches voted for each class.

---
## 6 · The appeal: three rounds  <small>44–60 min</small>

No new code. Every round is **predict, then run**.

▶ The pencil's eight ready-made parts.

In [ ]:
show_parts()

### Round 1 · Win the appeal
Get the judge to rule **cat, at 0.90 or more** (beyond reasonable doubt).
Swap, add or delete parts **one change at a time**, and call out your prediction before each run.

▶ Robby's cat. Edit the parts in this line.

In [ ]:
judge(cnn, face, round_ears, snout, eyes)          # Robby's cat: edit this line

**Question for class:** Which single change got closest? Why did it take two? *(Look at where it sees a pig.)*

**Takeaway:** every part adds evidence for some class, and the ruling goes to the class with the
most. The snout is strong pig evidence: you have to remove it, or outvote it with cat evidence.

### Round 2 · The smallest cat
Start from a full cat and **delete parts** until the judge stops saying cat.
What is the smallest set of parts it still rules a cat?

▶ A full cat. Delete parts from this line.

In [ ]:
judge(cnn, face, pointy_ears, eyes, nose, smile, whiskers)     # a full cat: delete parts

**Question for class:** Which part is in every small cat you found? Did you need the face?

**Takeaway:** the network never learned what a whole cat looks like, only which *parts* show up in
cat drawings. A cat, to this network, is pointy ears plus a little more.

### Round 3 · Cut it up, turn it over
▶ A real cat from the dataset, cut in half with the halves swapped. **Vote:** cat or not? Then run.

In [ ]:
judge(cnn, np.roll(X[296], 14, axis=0))            # the cat from section 2, top and bottom halves swapped

▶ The same cat, upside down. **Vote** again, then run.

In [ ]:
judge(cnn, X[296][::-1])

**Question for class:** Why is the cut-up cat still a cat, but the upside-down cat isn't?

**Takeaway:** `evidence = h.max(axis=(1, 2))` asks *whether* each feature appeared, not *where*, so
a cut-up cat still has its ears and whiskers. Turning it over changes the parts themselves: filters
that learned upright ears don't fire on upside-down ones. Shifts come free with convolution;
rotation you teach with data (**augmentation**, slide 75).

| what the CNN gives you | slides |
|---|---|
| the same filter at every position → equivariance to shifts | 11–12 |
| weight sharing → fewer parameters than a dense network | 26–31 |
| pooling and a global max → it doesn't care *where*: survives a nudge, fooled by a cut-up cat | 9–10, 63–65 |
| local receptive fields → a heat map for free | 42–45 |
| **not** rotation or scale: you add those with data augmentation | 75 |

**Next:** the famous architectures (slides 59–100): AlexNet, VGG, ResNet, U-Net.

*Data: Google Creative Lab, [The Quick, Draw! Dataset](https://github.com/googlecreativelab/quickdraw-dataset), CC BY 4.0.*